# Handling Multi-Agent Runs
Companion notebook for the **Handling Multi-Agent Runs** section of Chapter 7.

Tasks are usually non-preemptive once started, so a single queue lets one long task block everything behind it (head-of-line blocking). The fix: split work by priority/tenant into multiple queues, and let a fair scheduler pick across them.

In [1]:
from collections import deque

queues = {
    "high": deque(),
    "normal": deque(),
    "low": deque(),
}

tenant_active: dict[str, int] = {}
TENANT_LIMIT = 2


def pick_next():
    """Scan lanes high -> normal -> low; skip a tenant already at its concurrency limit."""
    for lane in ("high", "normal", "low"):
        q = queues[lane]
        for _ in range(len(q)):
            task = q[0]
            tenant = task["tenant"]
            if tenant_active.get(tenant, 0) < TENANT_LIMIT:
                q.popleft()
                tenant_active[tenant] = tenant_active.get(tenant, 0) + 1
                return task
            q.rotate(-1)
    return None

Load the queues with a busy tenant (`acme`, 5 tasks) and a quiet one (`globex`, 1 task) all in the same `normal` lane, plus one `high`-priority task. `pick_next()` should still let `globex` through promptly instead of starving behind `acme`'s backlog, because `acme` hits `TENANT_LIMIT` after 2 concurrent picks.

In [2]:
for i in range(5):
    queues["normal"].append({"id": f"acme-{i}", "tenant": "acme"})
queues["normal"].append({"id": "globex-0", "tenant": "globex"})
queues["high"].append({"id": "urgent-0", "tenant": "acme"})

picked = []
while (task := pick_next()) is not None:
    picked.append(task["id"])

print("Pick order:", picked)
print("tenant_active after draining what fits under the limit:", tenant_active)

Pick order: ['urgent-0', 'acme-0', 'globex-0']
tenant_active after draining what fits under the limit: {'acme': 2, 'globex': 1}


Notice `urgent-0` (in the `high` lane) is picked before any `normal`-lane task, and `globex-0` gets through even though it was queued behind several `acme` tasks -- once `acme` hits `TENANT_LIMIT=2`, `pick_next()` skips its remaining tasks and finds `globex`'s instead of blocking on `acme`'s backlog. In production you'd also wrap each shared dependency (model API, vector store, database) in its own concurrency cap and timeout, so one failing dependency can't cascade through the whole workflow.